# 03 — DSR & Market Access

Materialises the VPP registry and the bronze → silver → gold tables described in [`../specifications/03-dsr-market-access.md`](../specifications/03-dsr-market-access.md).

**Tables produced**
- `short_term_dim_dsr_assets`, `short_term_dim_dsr_owners`
- `short_term_bronze_submeter_telemetry`
- `short_term_silver_dsr_availability`
- `short_term_gold_dsr_bid_stack`, `short_term_gold_dsr_dispatch` (feeds the control tower), `short_term_gold_dsr_settlement`, `short_term_gold_dsr_summary`

Depends on **notebook 01** for the shared dimensions (`short_term_dim_zones`, `short_term_dim_intervals`). Demo materialises gold in batch over the same 15-minute interval grid; production would stream high-cardinality submeter telemetry. **UC comments:** [`uc_table_comments.py`](./uc_table_comments.py).

In [ ]:
# Databricks notebook source
# MAGIC %md
# MAGIC ## Preamble — catalog/schema, shared dimensions (from notebook 01)

# COMMAND ----------

import os
import math
import random
import datetime as dt

from pyspark.sql import functions as F
from pyspark.sql import Row

dbutils.widgets.text("catalog", os.environ.get("DEMO_UC_CATALOG", "energy_utilities"))
dbutils.widgets.text("schema", os.environ.get("DEMO_UC_SCHEMA", "energy_trading2"))

CATALOG = dbutils.widgets.get("catalog").strip() or "energy_utilities"
SCHEMA = dbutils.widgets.get("schema").strip() or "energy_trading2"

spark.sql(f"USE `{CATALOG}`.`{SCHEMA}`")

def fq(name: str) -> str:
    return f"`{CATALOG}`.`{SCHEMA}`.`{name}`"

random.seed(303)

# Read the shared time spine and zones owned by notebook 01.
intervals_df = spark.table(fq("short_term_dim_intervals"))
ivals = [(r.delivery_date, r.interval_start, r.interval_index) for r in
         intervals_df.select("delivery_date", "interval_start", "interval_index").orderBy("interval_start").collect()]
ZONES = [r.zone_code for r in spark.table(fq("short_term_dim_zones")).select("zone_code").collect()]
DELIVERY_DATES = sorted({d for d, _, _ in ivals})
LATEST_DATE = DELIVERY_DATES[-1]
NOW_INDEX = 56

def is_settled(day, idx0):
    if day < LATEST_DATE:
        return True
    if day == LATEST_DATE:
        return idx0 < NOW_INDEX
    return False

def solar_window(idx0):
    return 10 <= (idx0 * 15 // 60) < 16

def evening_peak(idx0):
    return 18 <= (idx0 * 15 // 60) < 22

print("zones:", ZONES, "| days:", [d.isoformat() for d in DELIVERY_DATES], "| intervals:", len(ivals))

In [ ]:
# MAGIC %md
# MAGIC ## 1. Registry — owners and VPP assets (~150 MW aggregated fleet)

# COMMAND ----------

owners = [
    Row(owner_id="OWN_IND_01", owner_name="Rheinwerk Industrie GmbH",  owner_segment="INDUSTRIAL",      share_pct=0.80, settlement_currency="EUR"),
    Row(owner_id="OWN_IND_02", owner_name="Nordstahl AG",              owner_segment="INDUSTRIAL",      share_pct=0.78, settlement_currency="EUR"),
    Row(owner_id="OWN_MOB_01", owner_name="VoltDrive Fleet Services",  owner_segment="MOBILITY",        share_pct=0.82, settlement_currency="EUR"),
    Row(owner_id="OWN_RES_01", owner_name="HeatPool Residential Agg.", owner_segment="RESIDENTIAL_AGG", share_pct=0.75, settlement_currency="EUR"),
    Row(owner_id="OWN_COM_01", owner_name="CityCentre Facilities",     owner_segment="COMMERCIAL",      share_pct=0.79, settlement_currency="EUR"),
    Row(owner_id="OWN_COM_02", owner_name="Logistik Park Süd",         owner_segment="COMMERCIAL",      share_pct=0.81, settlement_currency="EUR"),
]
(spark.createDataFrame(owners)
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(fq("short_term_dim_dsr_owners")))

ASSET_TYPES = {
    "INDUSTRIAL_LOAD": {"owners": ["OWN_IND_01", "OWN_IND_02"], "up": (0.8, 3.0), "down": (0.2, 0.8), "resp": 60,  "minrt": 15, "mkts": "AFRR,MFRR,INTRADAY"},
    "EV_DEPOT":        {"owners": ["OWN_MOB_01"],               "up": (0.3, 1.2), "down": (0.8, 2.5), "resp": 30,  "minrt": 5,  "mkts": "AFRR,INTRADAY"},
    "HEAT_PUMP":       {"owners": ["OWN_RES_01"],               "up": (0.1, 0.4), "down": (0.1, 0.5), "resp": 120, "minrt": 30, "mkts": "MFRR,INTRADAY"},
    "BTM_BATTERY":     {"owners": ["OWN_COM_01", "OWN_COM_02"], "up": (0.2, 1.0), "down": (0.2, 1.0), "resp": 10,  "minrt": 5,  "mkts": "AFRR,MFRR,INTRADAY"},
}

dsr_assets = []
target_mw, total_up = 150.0, 0.0
i = 0
# Round-robin asset types until we reach ~150 MW aggregate flex-up.
type_cycle = list(ASSET_TYPES.keys())
while total_up < target_mw and i < 400:
    atype = type_cycle[i % len(type_cycle)]
    cfg = ASSET_TYPES[atype]
    up = round(random.uniform(*cfg["up"]), 2)
    down = round(random.uniform(*cfg["down"]), 2)
    owner = random.choice(cfg["owners"])
    zone = random.choices(ZONES, weights=[5, 2, 2, 2, 1])[0]
    prefix = {"INDUSTRIAL_LOAD": "IND", "EV_DEPOT": "EVDEPOT", "HEAT_PUMP": "HP", "BTM_BATTERY": "BTMB"}[atype]
    dsr_assets.append(Row(
        dsr_asset_id=f"{prefix}_{zone}_{i:04d}",
        owner_id=owner,
        asset_type=atype,
        zone_code=zone,
        max_flex_up_mw=up,
        max_flex_down_mw=down,
        response_time_s=cfg["resp"],
        min_runtime_min=cfg["minrt"],
        prequalified_markets=cfg["mkts"],
    ))
    total_up += up
    i += 1

(spark.createDataFrame(dsr_assets)
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(fq("short_term_dim_dsr_assets")))

print("owners:", spark.table(fq("short_term_dim_dsr_owners")).count())
print("dsr_assets:", spark.table(fq("short_term_dim_dsr_assets")).count(), "| aggregate flex-up MW:", round(total_up, 1))

In [ ]:
# MAGIC %md
# MAGIC ## 2. Bronze submeter telemetry + Silver availability (Mosaic AI baseline)

# COMMAND ----------

assets = spark.table(fq("short_term_dim_dsr_assets")).collect()

def baseline_factor(atype: str, idx0: int) -> float:
    hour = idx0 * 15 / 60
    if atype == "INDUSTRIAL_LOAD":
        return 0.8 if 6 <= hour < 22 else 0.4
    if atype == "EV_DEPOT":
        return 0.7 if (hour < 7 or hour >= 18) else 0.25  # charges off-peak/evening
    if atype == "HEAT_PUMP":
        return 0.7 if (6 <= hour < 9 or 17 <= hour < 22) else 0.35
    return 0.1  # BTM battery near-zero net baseline

telemetry_rows, avail_rows = [], []
for day, start, idx1 in ivals:
    idx0 = idx1 - 1
    settled = is_settled(day, idx0)
    for a in assets:
        bf = baseline_factor(a.asset_type, idx0)
        baseline = round((a.max_flex_up_mw + a.max_flex_down_mw) * 0.5 * bf, 3)
        # Flex up = curtail/discharge capability; higher when load is high.
        up = round(a.max_flex_up_mw * min(1.0, 0.4 + bf), 3)
        # Flex down = absorb/charge capability; higher midday (soak solar) for EV/battery.
        soak = 1.0 if (a.asset_type in ("EV_DEPOT", "BTM_BATTERY") and solar_window(idx0)) else 0.5
        down = round(a.max_flex_down_mw * soak, 3)
        direction = "BOTH" if (up > 0 and down > 0) else ("UP" if up > 0 else ("DOWN" if down > 0 else "NONE"))
        avail_rows.append(Row(
            delivery_date=day,
            interval_start=start,
            dsr_asset_id=a.dsr_asset_id,
            owner_id=a.owner_id,
            asset_type=a.asset_type,
            zone_code=a.zone_code,
            baseline_mw=baseline,
            available_up_mw=up,
            available_down_mw=down,
            direction=direction,
            confidence=round(random.uniform(0.75, 0.98), 2),
        ))
        if settled:
            consumption = round(baseline * (1 + random.gauss(0, 0.05)), 3)
            telemetry_rows.append(Row(
                ingestion_ts=start,
                telemetry_ts=start,
                dsr_asset_id=a.dsr_asset_id,
                consumption_mw=consumption,
                baseline_mw=baseline,
                soc_pct=round(random.uniform(20, 90), 1) if a.asset_type == "BTM_BATTERY" else None,
                status=random.choices(["ONLINE", "OFFLINE"], weights=[97, 3])[0],
            ))

(spark.createDataFrame(avail_rows)
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(fq("short_term_silver_dsr_availability")))
(spark.createDataFrame(telemetry_rows)
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(fq("short_term_bronze_submeter_telemetry")))

print("availability:", spark.table(fq("short_term_silver_dsr_availability")).count())
print("telemetry:", spark.table(fq("short_term_bronze_submeter_telemetry")).count())

In [ ]:
# MAGIC %md
# MAGIC ## 3. Gold — prequalified bid stack + dispatch & verification

# COMMAND ----------

MIN_SIZE = {"AFRR": 1.0, "MFRR": 5.0, "INTRADAY": 0.1}
BID_PRICE = {"AFRR": 85.0, "MFRR": 70.0, "INTRADAY": 95.0}

# ---- Bid stack: explode each asset's prequalified markets, aggregate flex-up ----
av = (spark.table(fq("short_term_silver_dsr_availability"))
        .join(spark.table(fq("short_term_dim_dsr_assets")).select("dsr_asset_id", "prequalified_markets"),
              "dsr_asset_id", "left")
        .withColumn("market", F.explode(F.split(F.col("prequalified_markets"), ","))))

bid_stack = (av.groupBy("delivery_date", "interval_start", "market", "zone_code")
    .agg(F.round(F.sum("available_up_mw"), 2).alias("qualified_mw"),
         F.count("*").alias("n_assets"))
    .withColumn("min_size_mw",
        F.when(F.col("market") == "AFRR", F.lit(MIN_SIZE["AFRR"]))
         .when(F.col("market") == "MFRR", F.lit(MIN_SIZE["MFRR"]))
         .otherwise(F.lit(MIN_SIZE["INTRADAY"])))
    .withColumn("bid_price_eur_mwh",
        F.when(F.col("market") == "AFRR", F.lit(BID_PRICE["AFRR"]))
         .when(F.col("market") == "MFRR", F.lit(BID_PRICE["MFRR"]))
         .otherwise(F.lit(BID_PRICE["INTRADAY"])))
    .withColumn("prequalification_status",
        F.when(F.col("qualified_mw") >= F.col("min_size_mw"), F.lit("QUALIFIED"))
         .otherwise(F.lit("BELOW_MIN_SIZE")))
    .select("delivery_date", "interval_start", "market", "zone_code", "qualified_mw",
            "n_assets", "bid_price_eur_mwh", "prequalification_status", "min_size_mw"))

(bid_stack.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(fq("short_term_gold_dsr_bid_stack")))

# ---- Dispatch: triggered intervals only, from zone-interval flex aggregates ----
agg_rows = (spark.table(fq("short_term_silver_dsr_availability"))
    .groupBy("delivery_date", "interval_start", "zone_code")
    .agg(F.sum("available_up_mw").alias("up"), F.sum("available_down_mw").alias("down"))
    .collect())

dispatch_rows = []
for r in agg_rows:
    day, start, zone = r.delivery_date, r.interval_start, r.zone_code
    idx0 = int(round((start - dt.datetime.combine(day, dt.time(0, 0))).total_seconds() / 900.0))
    long_zone = (zone in ("DE", "NL"))
    if solar_window(idx0) and long_zone:
        market, trigger = "INTRADAY", "NEGATIVE_PRICE"
        dispatched = -round(min(r.down, r.down * 0.6), 2)  # absorb (charge / raise load)
        price = round(random.uniform(-40, -5), 2)
    elif evening_peak(idx0):
        market, trigger = "MFRR", "SCARCITY"
        dispatched = round(min(r.up, r.up * 0.7), 2)        # discharge / curtail
        price = round(random.uniform(120, 260), 2)
    elif random.random() < 0.06:
        market, trigger = "AFRR", "FREQUENCY"
        dispatched = round(random.choice([1, -1]) * min(r.up, r.up * 0.3), 2)
        price = round(random.uniform(60, 110), 2)
    else:
        continue
    ratio = round(random.uniform(0.88, 1.0), 3)
    delivered = round(dispatched * ratio, 2)
    status = "VERIFIED" if ratio >= 0.95 else "UNDER_DELIVERED"
    if not is_settled(day, idx0):
        status = "PENDING"
    dispatch_rows.append(Row(
        delivery_date=day,
        interval_start=start,
        market=market,
        zone_code=zone,
        signal_trigger=trigger,
        dispatched_mw=dispatched,
        delivered_mw=delivered,
        delivery_ratio=ratio,
        clearing_price_eur_mwh=price,
        verification_status=status,
    ))

(spark.createDataFrame(dispatch_rows)
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(fq("short_term_gold_dsr_dispatch")))

print("bid_stack:", spark.table(fq("short_term_gold_dsr_bid_stack")).count())
print("dispatch:", spark.table(fq("short_term_gold_dsr_dispatch")).count())

In [ ]:
# MAGIC %md
# MAGIC ## 4. Gold — per-owner settlement (Delta Sharing target) + daily summary

# COMMAND ----------

dispatch = spark.table(fq("short_term_gold_dsr_dispatch"))
assets = spark.table(fq("short_term_dim_dsr_assets"))
owners = spark.table(fq("short_term_dim_dsr_owners"))

# Revenue and energy per (day, zone) from verified delivery (0.25h per interval).
zone_rev = (dispatch
    .withColumn("energy_mwh", F.abs(F.col("delivered_mw")) * F.lit(0.25))
    .withColumn("revenue_eur", F.col("energy_mwh") * F.col("clearing_price_eur_mwh"))
    .groupBy("delivery_date", "zone_code")
    .agg(F.sum("revenue_eur").alias("zone_revenue_eur"),
         F.sum("energy_mwh").alias("zone_mwh")))

# Owner weight within each zone by prequalified flex-up.
owner_w = assets.groupBy("zone_code", "owner_id").agg(F.sum("max_flex_up_mw").alias("w"))
zone_tot = assets.groupBy("zone_code").agg(F.sum("max_flex_up_mw").alias("zone_tot"))
owner_share = (owner_w.join(zone_tot, "zone_code")
    .withColumn("zone_share", F.col("w") / F.col("zone_tot")))

owner_zone = (owner_share.join(zone_rev, "zone_code")
    .withColumn("owner_revenue_eur", F.col("zone_revenue_eur") * F.col("zone_share"))
    .withColumn("owner_mwh", F.col("zone_mwh") * F.col("zone_share")))

# Assets dispatched per owner per day (owner assets in zones that saw dispatch).
disp_zones = dispatch.select("delivery_date", "zone_code").distinct()
n_disp = (assets.join(disp_zones, "zone_code")
    .groupBy("delivery_date", "owner_id").agg(F.countDistinct("dsr_asset_id").alias("n_assets_dispatched")))

settlement = (owner_zone.groupBy("delivery_date", "owner_id")
    .agg(F.round(F.sum("owner_revenue_eur"), 2).alias("market_revenue_eur"),
         F.round(F.sum("owner_mwh"), 3).alias("total_delivered_mwh"))
    .join(owners.select("owner_id", "owner_name", "share_pct"), "owner_id", "left")
    .join(n_disp, ["delivery_date", "owner_id"], "left")
    .fillna(0, subset=["n_assets_dispatched"])
    .withColumn("owner_payment_eur", F.round(F.col("market_revenue_eur") * F.col("share_pct"), 2))
    .withColumn("aggregator_fee_eur", F.round(F.col("market_revenue_eur") - F.col("owner_payment_eur"), 2))
    .withColumn("settlement_status",
        F.when(F.col("delivery_date") < F.lit(LATEST_DATE), F.lit("SETTLED")).otherwise(F.lit("PENDING")))
    .select("delivery_date", "owner_id", "owner_name", "total_delivered_mwh", "market_revenue_eur",
            "owner_payment_eur", "aggregator_fee_eur", "n_assets_dispatched", "settlement_status"))

(settlement.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(fq("short_term_gold_dsr_settlement")))

# ---- Daily summary ----
interval_tot = (spark.table(fq("short_term_silver_dsr_availability"))
    .groupBy("delivery_date", "interval_start")
    .agg(F.sum("available_up_mw").alias("up"), F.sum("available_down_mw").alias("down"))
    .groupBy("delivery_date")
    .agg(F.round(F.max("up"), 1).alias("total_available_up_mw"),
         F.round(F.max("down"), 1).alias("total_available_down_mw")))

disp_tot = (dispatch
    .withColumn("energy_mwh", F.abs(F.col("delivered_mw")) * F.lit(0.25))
    .groupBy("delivery_date")
    .agg(F.round(F.sum("energy_mwh"), 2).alias("total_dispatched_mwh"),
         F.round(F.avg("delivery_ratio"), 3).alias("avg_delivery_ratio")))

set_tot = (settlement.groupBy("delivery_date")
    .agg(F.round(F.sum("market_revenue_eur"), 2).alias("total_market_revenue_eur"),
         F.round(F.sum("owner_payment_eur"), 2).alias("total_owner_payments_eur"),
         F.countDistinct("owner_id").alias("n_owners")))

# Count distinct fleet assets per day from silver availability (bronze telemetry has no delivery_date).
online = (spark.table(fq("short_term_silver_dsr_availability"))
    .groupBy("delivery_date")
    .agg(F.countDistinct("dsr_asset_id").alias("n_assets_online")))

summary = (interval_tot
    .join(disp_tot, "delivery_date", "left")
    .join(set_tot, "delivery_date", "left")
    .join(online, "delivery_date", "left")
    .fillna(0)
    .withColumn("snapshot_ts", F.to_timestamp(F.concat(F.col("delivery_date").cast("string"), F.lit(" 14:00:00"))))
    .withColumn("headline",
        F.when(F.col("total_dispatched_mwh") > 0,
               F.concat(F.lit("VPP active — "), F.round(F.col("total_dispatched_mwh"), 0).cast("int").cast("string"),
                        F.lit(" MWh dispatched, €"), F.round(F.col("total_market_revenue_eur"), 0).cast("int").cast("string"), F.lit(" captured")))
         .otherwise(F.lit("Fleet on standby — no dispatch signals")))
    .select("delivery_date", "snapshot_ts", "headline", "total_available_up_mw", "total_available_down_mw",
            "total_dispatched_mwh", "total_market_revenue_eur", "total_owner_payments_eur",
            "avg_delivery_ratio", "n_assets_online", "n_owners"))

(summary.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(fq("short_term_gold_dsr_summary")))

display(spark.table(fq("short_term_gold_dsr_summary")).orderBy(F.col("delivery_date").desc()))

In [ ]:
# MAGIC %md
# MAGIC ## 5. Row counts + Unity Catalog comments

# COMMAND ----------

for t in [
    "short_term_dim_dsr_owners",
    "short_term_dim_dsr_assets",
    "short_term_bronze_submeter_telemetry",
    "short_term_silver_dsr_availability",
    "short_term_gold_dsr_bid_stack",
    "short_term_gold_dsr_dispatch",
    "short_term_gold_dsr_settlement",
    "short_term_gold_dsr_summary",
]:
    print(f"  {t:42s}  {spark.table(fq(t)).count():>10,} rows")

# COMMAND ----------

from pathlib import Path

_uc_paths = []
try:
    _nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    _uc_paths.append(Path(_nb).parent / "uc_table_comments.py")
except Exception:
    pass
_uc_paths.append(Path.cwd() / "uc_table_comments.py")

_uc_py = next((p for p in _uc_paths if p.is_file()), None)
if _uc_py is None:
    raise FileNotFoundError("uc_table_comments.py not found next to this notebook.")

exec(_uc_py.read_text(), globals())
apply_short_term_notebook_03_comments(spark, CATALOG, SCHEMA)
print("UC comments applied for notebook 03.")